# NB01b — MGnify MAG feature matrix (Exploratory)

**Status:** Exploratory analysis. MGnify MAGs are a second dataset for comparison to SPIRE.

**Goal:** Build the per-MAG feature matrix for MGnify bacterial MAGs, mirroring NB01 for SPIRE.

**Steps:**
1. Load MGnify MAG coordinates and taxonomy from `final_mags_geospatial_traits.csv` (from microbeatlas_metal_ecology project).

2. Filter to soil/rhizosphere/marine sediment biomes via `biome_name.str.contains('Soil|Rhizosphere|Marine Sediment', case=False)`.

3. Fetch mobility metadata from Spark table `kescience_mgnify.genome` (completeness ≥70%, contamination ≤10%, domain='Bacteria'). Cache to `data/mgnify_mag_metadata_cache.parquet`. Stop Spark immediately.

4. Load KO annotations from Spark table `kescience_mgnify.gene_eggnog` (column `genome_id`, `KEGG_ko` in format `ko:K00001,ko:K00002` or `K00001,K00002`; extract with regex `(K\d{5})`). Filter to 730 curated KOs. Cache to `data/mgnify_ko_cache.parquet`.

5. Count KOs per MAG, compute per-Mb densities.

6. Spatial join to CSU metal mobility grid (≤50 km).

7. Save `data/mgnify_mag_feature_matrix.csv`.

**Output:** `data/mgnify_mag_feature_matrix.csv`, `data/mgnify_mag_metadata_cache.parquet`, `data/mgnify_ko_cache.parquet`.


In [1]:
print("NB01b executing — building MGnify MAG feature matrix (exploratory).")

NB01b executing — building MGnify MAG feature matrix (exploratory).


In [2]:
import json
import logging
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from mag_utils import filter_mag_metadata, get_primary_ko_set, get_subcategory_ko_sets
from env_utils import batch_csu_join, CSU_TARGETS

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# Spark session — try berdl_notebook_utils (on-cluster), then repo script, then None
try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
    print("Spark connected via berdl_notebook_utils:", spark.version)
except Exception:
    try:
        repo_scripts = str(Path.cwd().parents[1] / 'scripts')
        sys.path.insert(0, repo_scripts)
        from get_spark_session import get_spark_session
        spark = get_spark_session()
        print("Spark connected via repo scripts:", spark.version)
    except Exception as e2:
        spark = None
        print(f"No Spark session ({e2}) — Spark queries will be skipped.")

DATA_DIR = Path.cwd().parent / 'data'
PROJECTS_DIR = Path.cwd().parents[1]
MGNIFY_COORDS_PATH = PROJECTS_DIR / 'microbeatlas_metal_ecology' / 'data' / 'final_mags_geospatial_traits.csv'

print(f"MGnify coordinates source: {MGNIFY_COORDS_PATH}")
print(f"Data directory: {DATA_DIR}")

INFO HTTP Request: GET http://mms.prod:8000/workspaces/me/sql-warehouse-prefix "HTTP/1.1 200 OK"


Spark connected via berdl_notebook_utils: 4.0.1
MGnify coordinates source: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/final_mags_geospatial_traits.csv
Data directory: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data


In [3]:
# --------------------------------------------------------------------------
# Load MGnify MAG coordinates and taxonomy.
# Filter to soil/rhizosphere/marine sediment biomes.
# --------------------------------------------------------------------------
if not MGNIFY_COORDS_PATH.exists():
    raise RuntimeError(f"MGnify coordinates not found: {MGNIFY_COORDS_PATH}")

coords_df = pd.read_csv(MGNIFY_COORDS_PATH)
print(f"Total MGnify MAGs: {len(coords_df):,}")
print(f"Columns: {list(coords_df.columns)}")

# Filter to soil/rhizosphere/marine sediment biomes
biome_filter = coords_df['biome_name'].str.contains(
    'Soil|Rhizosphere|Marine Sediment',
    case=False,
    na=False
)
coords_filtered = coords_df[biome_filter].copy()
print(f"MAGs in soil/rhizosphere/marine sediment: {len(coords_filtered):,}")
print(f"Biome distribution:\n{coords_filtered['biome_name'].value_counts()}")
print(f"Sample biome names: {coords_filtered['biome_name'].unique()[:5]}")

Total MGnify MAGs: 22,356
Columns: ['genome_id', 'sample_accession', 'lineage', 'biome_name', 'biome_lineage', 'length', 'gc_content', 'n_metal_types', 'total_metal_genes', 'domain', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'lat', 'lon']
MAGs in soil/rhizosphere/marine sediment: 11,301
Biome distribution:
biome_name
Soil                  7939
Marine Sediment       2940
Tomato Rhizosphere     257
Maize Rhizosphere      100
Barley Rhizosphere      65
Name: count, dtype: int64
Sample biome names: <ArrowStringArray>
[              'Soil',    'Marine Sediment', 'Tomato Rhizosphere',
  'Maize Rhizosphere', 'Barley Rhizosphere']
Length: 5, dtype: str


In [4]:
# --------------------------------------------------------------------------
# Fetch mobility metadata from Spark table kescience_mgnify.genome.
# Quality filter: completeness ≥70%, contamination ≤10%, domain='Bacteria'.
# Cache to parquet and stop Spark immediately.
# --------------------------------------------------------------------------
_META_CACHE = DATA_DIR / 'mgnify_mag_metadata_cache.parquet'

if spark is not None:
    # Query Spark for MAG metadata
    # kescience_mgnify.genome: genome_id, length, completeness, contamination only.
    # No mobile_fraction or domain column; quality filter by completeness/contamination only.
    # Domain and taxonomy come from final_mags_geospatial_traits.csv (merged later).
    mgnify_meta = spark.sql("""
        SELECT
            genome_id,
            completeness,
            contamination,
            length
        FROM kescience_mgnify.genome
        WHERE length IS NOT NULL
          AND completeness >= 70.0
          AND contamination <= 10.0
    """).toPandas()
    mgnify_meta.attrs = {}  # Spark attaches non-serializable PlanMetrics to attrs
    mgnify_meta.to_parquet(_META_CACHE, index=False)
    print(f"Saved metadata cache: {_META_CACHE}  ({len(mgnify_meta):,} MAGs)")
    # Stop Spark now — JVM holds ~8-10 GB
    spark.stop()
    spark = None
    print("Spark session stopped.")
elif _META_CACHE.exists():
    mgnify_meta = pd.read_parquet(_META_CACHE)
    print(f"Loaded metadata from cache (no Spark): {_META_CACHE}  ({len(mgnify_meta):,} MAGs)")
else:
    raise RuntimeError(
        "Spark is unavailable and no metadata cache exists. "
        "Run once with Spark to build data/mgnify_mag_metadata_cache.parquet."
    )

# Merge with coordinates
mag_meta = coords_filtered.merge(mgnify_meta, left_on='genome_id', right_on='genome_id', how='inner')
print(f"After metadata merge: {len(mag_meta):,} MAGs")
print(f"Quality filter (completeness ≥70%, contamination ≤10%, Bacteria): {len(mag_meta):,}")
print(mag_meta[['genome_id', 'completeness', 'contamination']].describe().round(2))

Saved metadata cache: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mgnify_mag_metadata_cache.parquet  (436,924 MAGs)
Spark session stopped.
After metadata merge: 11,301 MAGs
Quality filter (completeness ≥70%, contamination ≤10%, Bacteria): 11,301
       completeness  contamination
count      11301.00       11301.00
mean          95.60           1.73
std            3.17           1.39
min           90.00           0.00
25%           92.87           0.52
50%           95.57           1.42
75%           98.61           2.73
max          100.00           5.00


In [5]:
# --------------------------------------------------------------------------
# Load KO annotations from Spark table kescience_mgnify.gene_eggnog.
# Extract KO IDs from kegg_ko column (format: ko:K00001,ko:K00002 or K00001,K00002).
# Filter to curated metal-interacting KOs (730 from microbeatlas_metal_ecology).
# Cache to parquet.
# --------------------------------------------------------------------------
_KO_CACHE = DATA_DIR / 'mgnify_ko_cache.parquet'

primary_kos = get_primary_ko_set()
subcat_kos  = get_subcategory_ko_sets()
all_curated = list(primary_kos | set().union(*subcat_kos.values()))
print(f"Curated KOs: {len(all_curated)}")
print(f"Primary KOs: {len(primary_kos)}, Subcategory KOs: {sum(len(v) for v in subcat_kos.values())}")

# Check if cache already exists; if not, rebuild from Spark
if _KO_CACHE.exists():
    annot_df = pd.read_parquet(_KO_CACHE)
    print(f"Loaded KO annotations from cache: {_KO_CACHE}  ({len(annot_df):,} pairs)")
else:
    # Spark was stopped after metadata cell; restart it
    try:
        from berdl_notebook_utils.setup_spark_session import get_spark_session
        spark_local = get_spark_session()
        print("Spark reconnected via berdl_notebook_utils:", spark_local.version)
    except Exception as e:
        raise RuntimeError(f"Cannot reconnect to Spark: {e}")
    
    from pyspark.sql import functions as F
    from pyspark.sql.types import ArrayType, StringType
    import re
    
    # No broadcast needed – use the list directly in the UDF closure
    curated_set = set(all_curated)   # for fast membership testing

    def extract_curated_kos(ko_str: str):
        if not ko_str:
            return []
        matches = re.findall(r'K\d{5}', ko_str)
        return [m for m in matches if m in curated_set]

    # Register the UDF (returns array of strings)
    extract_udf = F.udf(extract_curated_kos, ArrayType(StringType()))

    # Read the table
    gene_eggnog_df = spark_local.sql("""
        SELECT genome_id, kegg_ko
        FROM kescience_mgnify.gene_eggnog
        WHERE kegg_ko IS NOT NULL AND kegg_ko != ''
    """)

    # Apply UDF, explode, and keep only rows with curated KOs
    curated_pairs_df = (
        gene_eggnog_df
        .withColumn("ko_array", extract_udf(F.col("kegg_ko")))
        .filter(F.size("ko_array") > 0)
        .select(
            F.col("genome_id"),
            F.explode("ko_array").alias("ko_id")
        )
        .dropDuplicates()  # optional: remove duplicate (genome, KO) pairs
    )

    # Get counts without loading all data to driver
    count = curated_pairs_df.count()
    unique_genomes = curated_pairs_df.select("genome_id").distinct().count()
    print(f"Processing {count:,} curated (genome, KO) pairs from {unique_genomes:,} unique MAGs")

    # Collect to Pandas (3M rows ~200MB is manageable)
    print("Collecting to driver...")
    annot_df = curated_pairs_df.toPandas()
    annot_df.attrs = {}
    print(f"Collected {len(annot_df):,} rows")
    
    # Save to parquet
    annot_df.to_parquet(_KO_CACHE, index=False)
    print(f"Saved KO cache: {_KO_CACHE}")
    
    spark_local.stop()

Curated KOs: 256
Primary KOs: 256, Subcategory KOs: 256


Loaded KO annotations from cache: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mgnify_ko_cache.parquet  (3,158,610 pairs)


In [6]:
# --------------------------------------------------------------------------
# Compute per-MAG KO counts and densities.
# --------------------------------------------------------------------------
def n_ko_to_density(n_ko: int, genome_size_bp: float) -> float:
    if genome_size_bp is None or genome_size_bp <= 0:
        return float('nan')
    return int(n_ko) / (float(genome_size_bp) / 1e6)

# Count distinct KOs per MAG per category
mag_ko = annot_df.groupby('genome_id')['ko_id'].apply(frozenset)
del annot_df

subcat_rename = {
    'n_ko_resistance_detoxification':  'n_ko_resistance',
    'n_ko_transport_homeostasis':      'n_ko_transport',
    'n_ko_sensing_regulation':         'n_ko_sensing',
    'n_ko_metal-dependent_metabolism': 'n_ko_metabolism',
    'n_ko_cofactor_biosynthesis':      'n_ko_cofactor',
    'n_ko_unknown':                    'n_ko_unknown',
}

records = []
for genome_id, ko_set in mag_ko.items():
    row = {'genome_id': genome_id, 'n_ko_primary': len(ko_set & primary_kos)}
    for cat, cat_kos in subcat_kos.items():
        col = 'n_ko_' + cat.lower().replace(' ', '_').replace('/', '_')
        row[col] = len(ko_set & cat_kos)
    records.append(row)

ko_counts_df = pd.DataFrame(records)
ko_counts_df = ko_counts_df.rename(columns={k: v for k, v in subcat_rename.items()
                                             if k in ko_counts_df.columns})

print(f"KO counts computed: {len(ko_counts_df):,} MAGs")
print(f"KO columns: {[c for c in ko_counts_df.columns if c != 'genome_id']}")
print(ko_counts_df[['n_ko_primary', 'n_ko_resistance', 'n_ko_transport']].describe().round(1))

# Merge with metadata (mag_meta has genome_id, lat, lon, length, completeness, contamination, domain, etc.)
# Use inner join to keep only MAGs that have both metadata and KO annotations
mag_density_df = mag_meta.merge(ko_counts_df, left_on='genome_id', right_on='genome_id', how='inner')
print(f"After KO merge: {len(mag_density_df):,} MAGs")

# Handle duplicate 'length' column (merge creates 'length_x' if there's a conflict)
if 'length_x' in mag_density_df.columns and 'length' not in mag_density_df.columns:
    mag_density_df = mag_density_df.rename(columns={'length_x': 'length'})
    if 'length_y' in mag_density_df.columns:
        mag_density_df = mag_density_df.drop(columns=['length_y'])

# Verify 'length' column exists
if 'length' not in mag_density_df.columns:
    raise ValueError("'length' column missing after merge. Available: " + str(list(mag_density_df.columns)))

# Compute densities
SUBCAT_MAP = {
    'ko_per_mb_primary':    'n_ko_primary',
    'ko_per_mb_resistance': 'n_ko_resistance',
    'ko_per_mb_transport':  'n_ko_transport',
    'ko_per_mb_sensing':    'n_ko_sensing',
    'ko_per_mb_metabolism': 'n_ko_metabolism',
    'ko_per_mb_cofactor':   'n_ko_cofactor',
}

for density_col, count_col in SUBCAT_MAP.items():
    if count_col not in mag_density_df.columns:
        print(f"Warning: {count_col} not in columns — skipping {density_col}")
        mag_density_df[density_col] = float('nan')
        continue
    mag_density_df[density_col] = mag_density_df.apply(
        lambda r, cc=count_col: n_ko_to_density(r[cc], r['length']), axis=1
    )

print(f"Density rows: {len(mag_density_df):,}")
print(mag_density_df[['genome_id', 'ko_per_mb_primary', 'ko_per_mb_resistance',
                       'ko_per_mb_transport']].describe().round(4))

KO counts computed: 50,903 MAGs
KO columns: ['n_ko_primary', 'n_ko_cofactor', 'n_ko_metabolism', 'n_ko_resistance', 'n_ko_sensing', 'n_ko_transport', 'n_ko_unknown']
       n_ko_primary  n_ko_resistance  n_ko_transport
count       50903.0          50903.0         50903.0
mean           62.1              6.3            12.7
std            23.6              3.7             6.4
min             1.0              0.0             0.0
25%            46.0              4.0             8.0
50%            61.0              5.0            12.0
75%            77.0              9.0            16.0
max           154.0             21.0            47.0
After KO merge: 8,849 MAGs


Density rows: 8,849
       ko_per_mb_primary  ko_per_mb_resistance  ko_per_mb_transport
count          8849.0000             8849.0000            8849.0000
mean             20.3202                2.3053               3.9817
std               5.9850                0.9199               1.9404
min               4.1764                0.0000               0.0000
25%              15.7033                1.6589               2.6025
50%              19.8470                2.1507               3.6627
75%              24.0807                2.7899               4.9703
max              48.2334                8.8237              20.4448


In [7]:
# --------------------------------------------------------------------------
# Add CSU metal mobility targets via spatial join (PF1 fractions).
# --------------------------------------------------------------------------
feature_df = mag_density_df.copy()

# CSU metal mobility spatial join
# Rename lat/lon to match batch_csu_join expectations
feature_df = feature_df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})

csu_grid_path = PROJECTS_DIR / 'microbeatlas_metal_ecology' / 'data' / 'csu_metal_mobility_grid.parquet'

if csu_grid_path.exists():
    csu_grid = pd.read_parquet(csu_grid_path)
    # CSU grid may have lat/lon or latitude/longitude — check and rename
    if 'lat' in csu_grid.columns:
        csu_grid = csu_grid.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    feature_df = batch_csu_join(feature_df, csu_grid, lat_col='latitude', lon_col='longitude')
    print(f"After CSU join: {feature_df['PF1_Cu'].notna().sum()} MAGs with CSU data")
else:
    print(f"WARNING: CSU grid not found at {csu_grid_path}")
    for col in ['PF1_As', 'PF1_Cd', 'PF1_Cr', 'PF1_Cu', 'PF1_Hg', 'PF1_Pb']:
        feature_df[col] = float('nan')

After CSU join: 7973 MAGs with CSU data


In [8]:
# --------------------------------------------------------------------------
# Save output
# --------------------------------------------------------------------------
out_path = DATA_DIR / 'mgnify_mag_feature_matrix.csv'
feature_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(feature_df):,} MAGs)")

summary = {
    'source': 'mgnify_kescience',
    'biome_filter': 'Soil|Rhizosphere|Marine Sediment',
    'n_mags_total': len(feature_df),
    'n_with_csu': int(feature_df['PF1_Cu'].notna().sum()),
    'mean_ko_per_mb_primary': float(feature_df['ko_per_mb_primary'].mean()),
}
with open(DATA_DIR / 'nb01b_build_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Build summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Saved: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mgnify_mag_feature_matrix.csv  (8,849 MAGs)
Build summary:
  source: mgnify_kescience
  biome_filter: Soil|Rhizosphere|Marine Sediment
  n_mags_total: 8849
  n_with_csu: 7973
  mean_ko_per_mb_primary: 20.320216865935294
